<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/1splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 基于rating2inter.ipynb生成的5-core交互图，Train/Validation/Test data splitting
- Based on generated interactions, perform data splitting


In [1]:
import os, csv
import pandas as pd

In [ ]:
# os.chdir('/home/enoche/MMRec/Sports14')
# os.getcwd()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = "/content/drive/MyDrive/Classroom/HTTT_Projects/CD HTTT TM/MMRec/data"

In [4]:
!cp "{PATH}/sports14-indexed.inter" "."

## 直接加载现成的, Load interactions

In [5]:
rslt_file = 'sports14-indexed.inter'
df = pd.read_csv(rslt_file, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1390694400,0
1,1,0,5.0,1328140800,0
2,2,0,4.0,1330387200,0
3,3,0,4.0,1328400000,0


In [6]:
import random
import numpy as np

In [7]:
df = df.sample(frac=1).reset_index(drop=True)

df.sort_values(by=['userID'], inplace=True)
df[:20]

,userID,itemID,rating,timestamp,x_label
249292,0,17787,3.0,1391990400,1
177599,0,5458,5.0,1405123200,2
90586,0,3369,5.0,1405123200,2
2926,0,11981,2.0,1390694400,0
68847,0,13372,5.0,1391990400,1
131004,0,15852,5.0,1390694400,0
69240,0,0,5.0,1390694400,0
261548,0,3327,3.0,1391990400,1
54250,1,7215,5.0,1285372800,0
274941,1,3087,5.0,1293494400,0


In [15]:
uid_field, iid_field = 'userID', 'itemID'

uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [17787, 5458, 3369, 11981, 13372, 15852, 0, 3327]),
 (1,
  [7215,
   3087,
   7169,
   0,
   2374,
   4123,
   4281,
   2322,
   14212,
   13468,
   11502,
   15278,
   9198,
   1542,
   5044,
   8802,
   6677,
   6554,
   15249]),
 (2,
  [10837,
   14657,
   15360,
   9254,
   6298,
   8776,
   2950,
   4155,
   0,
   3661,
   7950,
   500,
   9841,
   14242,
   11360,
   14699,
   9250,
   3011,
   10218,
   15075,
   6445,
   1114,
   14212])]

In [19]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [ ]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())

for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:100]

[0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 2,
 2,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [ ]:
df['x_label'] = new_label
df[:20]

,userID,itemID,rating,timestamp,x_label
154667,0,11981,2.0,1390694400,0
295557,0,15852,5.0,1390694400,0
189316,0,17787,3.0,1391990400,0
151302,0,0,5.0,1390694400,0
1820,0,3369,5.0,1405123200,0
60040,0,13372,5.0,1391990400,0
199192,0,5458,5.0,1405123200,1
163234,0,3327,3.0,1391990400,2
60837,1,2322,5.0,1337212800,0
233786,1,4123,5.0,1354838400,0


In [ ]:
rslt_file[:-6]

'beauty14-indexed'

In [ ]:
new_labeled_file = rslt_file[:-6] + '-v4.inter'
df.to_csv(os.path.join('./', new_labeled_file), sep='\t', index=False)
print('done!!!')

done!!!


## Reload

In [ ]:
indexed_df = pd.read_csv(new_labeled_file, sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:20]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,0,11981,2.0,1390694400,0
1,0,15852,5.0,1390694400,0
2,0,17787,3.0,1391990400,0
3,0,0,5.0,1390694400,0
4,0,3369,5.0,1405123200,0
5,0,13372,5.0,1391990400,0
6,0,5458,5.0,1405123200,1
7,0,3327,3.0,1391990400,2
8,1,2322,5.0,1337212800,0
9,1,4123,5.0,1354838400,0


In [ ]:
u_id_str, i_id_str = 'userID', 'itemID'
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 35598
# of unique courses: 18357
min/max of unique learners: 0/35597
min/max of unique courses: 0/18356
